# 🚗 Traffic Accident Data Analysis
### Task-05 | Data Science Internship

**Dataset:** US Accidents (March 2023) — Kaggle  
**Objective:** Analyze accident patterns related to time, weather, severity, and geography to uncover key insights about road safety in the US.

---

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Global plot style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 12

print('✅ Libraries imported successfully.')

## 📂 Step 2: Load & Preprocess Data

In [ ]:
# Load first 50,000 rows for efficient processing
DATA_PATH = 'US_Accidents_March23.csv'
SAMPLE_SIZE = 50_000

print(f'Loading {SAMPLE_SIZE:,} rows from dataset...')
df = pd.read_csv(DATA_PATH, nrows=SAMPLE_SIZE, low_memory=False)

print(f'✅ Data loaded successfully!')
print(f'   Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# Preview the dataset
df.head(3)

In [ ]:
# --- Preprocessing ---

# 1. Convert Start_Time to datetime
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')

# 2. Extract time-based features
df['Hour']       = df['Start_Time'].dt.hour
df['Month']      = df['Start_Time'].dt.month
df['Month_Name'] = df['Start_Time'].dt.strftime('%b')
df['Day_Name']   = df['Start_Time'].dt.strftime('%a')
df['Year']       = df['Start_Time'].dt.year

# 3. Drop rows where key columns are missing
df.dropna(subset=['Start_Time', 'Severity', 'State', 'Weather_Condition'], inplace=True)

print('✅ Preprocessing complete.')
print(f'   Remaining rows after cleaning: {len(df):,}')

In [ ]:
# Summary statistics
print('--- Dataset Info ---')
df.info()
print()
print('--- Null Value Counts (Top 10 columns) ---')
print(df.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# Descriptive statistics for numerical columns
df[['Severity', 'Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Distance(mi)']].describe().round(2)

---
## ⏰ Step 3: Time-Based Analysis

In [ ]:
# --- 3a. Accidents by Hour of Day ---

hour_counts = df['Hour'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(hour_counts.index, hour_counts.values,
              color=sns.color_palette('rocket', 24), edgecolor='white', linewidth=0.5)

# Highlight peak hour
peak_hour = hour_counts.idxmax()
bars[peak_hour].set_edgecolor('gold')
bars[peak_hour].set_linewidth(2.5)

ax.set_xlabel('Hour of Day (0 = Midnight)', fontsize=12)
ax.set_ylabel('Number of Accidents', fontsize=12)
ax.set_title('🕐 Accident Frequency by Hour of Day', fontsize=15, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.annotate(f'Peak: {peak_hour}:00 h', xy=(peak_hour, hour_counts[peak_hour]),
            xytext=(peak_hour + 1.5, hour_counts[peak_hour] * 0.95),
            arrowprops=dict(arrowstyle='->', color='gold'), color='gold', fontsize=11)
plt.tight_layout()
plt.savefig('plot_hour.png', bbox_inches='tight')
plt.show()

print(f'📌 Peak accident hour: {peak_hour}:00 — {hour_counts[peak_hour]:,} accidents')

In [ ]:
# --- 3b. Accidents by Month ---

month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_counts = df.groupby('Month_Name').size().reindex(month_order)

fig, ax = plt.subplots(figsize=(13, 5))
sns.barplot(x=month_counts.index, y=month_counts.values, palette='coolwarm', ax=ax)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Number of Accidents', fontsize=12)
ax.set_title('📅 Accident Frequency by Month', fontsize=15, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('plot_month.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 3c. Accidents by Day of Week ---

day_order = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
day_counts = df.groupby('Day_Name').size().reindex(day_order)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=day_counts.index, y=day_counts.values, palette='viridis', ax=ax)
ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Number of Accidents', fontsize=12)
ax.set_title('📆 Accident Frequency by Day of Week', fontsize=15, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('plot_dayofweek.png', bbox_inches='tight')
plt.show()

---
## 🌦️ Step 4: Weather Impact Analysis

In [ ]:
# --- 4a. Top 10 Weather Conditions ---

top_weather = df['Weather_Condition'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(13, 6))
bars = sns.barplot(x=top_weather.values, y=top_weather.index,
                   palette='Blues_r', orient='h', ax=ax)
ax.set_xlabel('Number of Accidents', fontsize=12)
ax.set_ylabel('Weather Condition', fontsize=12)
ax.set_title('🌡️ Top 10 Weather Conditions During Accidents', fontsize=15, fontweight='bold')

# Annotate bars
for i, v in enumerate(top_weather.values):
    ax.text(v + 30, i, f'{v:,}', va='center', fontsize=10)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('plot_weather.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 4b. Day vs Night Accidents ---

day_night = df['Sunrise_Sunset'].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    day_night.values, labels=day_night.index,
    autopct='%1.1f%%', startangle=140,
    colors=['#FFC107', '#3F51B5'],
    explode=[0.03, 0.03], shadow=True
)
for t in autotexts:
    t.set_fontsize(13)
    t.set_fontweight('bold')
ax.set_title('☀️🌙 Accidents: Day vs Night', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_daynight.png', bbox_inches='tight')
plt.show()

---
## 🚦 Step 5: Severity Analysis

In [ ]:
# --- 5a. Severity Distribution ---

severity_counts = df['Severity'].value_counts().sort_index()
severity_labels = {1: 'Level 1\n(Minor)', 2: 'Level 2\n(Moderate)',
                   3: 'Level 3\n(Serious)', 4: 'Level 4\n(Critical)'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
colors = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C']
bars = ax1.bar([severity_labels[k] for k in severity_counts.index],
               severity_counts.values, color=colors, edgecolor='white')
ax1.set_xlabel('Severity Level', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Accident Severity Distribution', fontsize=14, fontweight='bold')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, severity_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Pie chart
ax2.pie(severity_counts.values,
        labels=[severity_labels[k] for k in severity_counts.index],
        autopct='%1.1f%%', colors=colors, startangle=90,
        explode=[0.03]*len(severity_counts), shadow=True)
ax2.set_title('Severity Share (%)', fontsize=14, fontweight='bold')

plt.suptitle('🚦 Accident Severity Analysis', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_severity.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5b. Weather Condition vs Severity (heatmap) ---

top10_weather = df['Weather_Condition'].value_counts().head(10).index.tolist()
heat_df = df[df['Weather_Condition'].isin(top10_weather)]
pivot = heat_df.pivot_table(index='Weather_Condition', columns='Severity',
                             values='ID', aggfunc='count', fill_value=0)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('🌦️ Weather Condition vs Severity Heatmap', fontsize=15, fontweight='bold')
ax.set_xlabel('Severity Level', fontsize=12)
ax.set_ylabel('Weather Condition', fontsize=12)
plt.tight_layout()
plt.savefig('plot_weather_severity_heatmap.png', bbox_inches='tight')
plt.show()

---
## 🗺️ Step 6: Geographic Analysis

In [ ]:
# --- 6a. Top 15 States by Accident Count ---

top_states = df['State'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(13, 6))
bars = sns.barplot(x=top_states.index, y=top_states.values,
                   palette='magma', ax=ax)
ax.set_xlabel('US State', fontsize=12)
ax.set_ylabel('Number of Accidents', fontsize=12)
ax.set_title('🏛️ Top 15 States with Most Accidents', fontsize=15, fontweight='bold')

for i, v in enumerate(top_states.values):
    ax.text(i, v + 30, f'{v:,}', ha='center', fontsize=9, fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('plot_states.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 6b. Geographic Scatter Plot (Accident Hotspots) ---

geo_df = df.dropna(subset=['Start_Lat', 'Start_Lng'])
# Sample 10,000 points for readability
geo_sample = geo_df.sample(n=min(10_000, len(geo_df)), random_state=42)

fig, ax = plt.subplots(figsize=(16, 8))
scatter = ax.scatter(
    geo_sample['Start_Lng'], geo_sample['Start_Lat'],
    c=geo_sample['Severity'], cmap='RdYlGn_r',
    alpha=0.35, s=4, linewidths=0
)
cbar = plt.colorbar(scatter, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label('Severity Level', fontsize=11)

ax.set_xlim(-130, -65)
ax.set_ylim(23, 52)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('📍 US Accident Hotspots — Scatter Map (Colored by Severity)', fontsize=15, fontweight='bold')
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#0f0f1a')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.title.set_color('white')

plt.tight_layout()
plt.savefig('plot_hotspots.png', bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# --- 6c. Top 10 Cities by Accident Count ---

top_cities = df['City'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=top_cities.index, y=top_cities.values, palette='cubehelix', ax=ax)
ax.set_xlabel('City', fontsize=12)
ax.set_ylabel('Number of Accidents', fontsize=12)
ax.set_title('🏙️ Top 10 Cities with Most Accidents', fontsize=15, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(top_cities.values):
    ax.text(i, v + 5, f'{v:,}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('plot_cities.png', bbox_inches='tight')
plt.show()

---
## 🔬 Step 7: Additional Insights

In [ ]:
# --- 7a. Correlation Heatmap of Numerical Features ---

num_cols = ['Severity', 'Distance(mi)', 'Temperature(F)',
             'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
             'Wind_Speed(mph)', 'Precipitation(in)']

corr_matrix = df[num_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
             cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('📊 Correlation Heatmap — Numerical Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 7b. Severity vs Temperature (Box Plot) ---

temp_df = df[df['Temperature(F)'].between(-30, 130)]

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(x='Severity', y='Temperature(F)', data=temp_df,
             palette=['#4CAF50','#FFC107','#FF5722','#B71C1C'], ax=ax)
ax.set_xlabel('Severity Level', fontsize=12)
ax.set_ylabel('Temperature (°F)', fontsize=12)
ax.set_title('🌡️ Severity vs Temperature Distribution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_severity_temp.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 7c. Road Feature Involvement ---

road_features = ['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction',
                  'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop',
                  'Traffic_Calming', 'Traffic_Signal']

feature_counts = df[road_features].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
feature_counts.plot(kind='barh', color=sns.color_palette('Set2', len(feature_counts)), ax=ax)
ax.set_xlabel('Number of Accidents', fontsize=12)
ax.set_title('🛣️ Road Feature Involvement in Accidents', fontsize=15, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for i, v in enumerate(feature_counts.values):
    ax.text(v + 10, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('plot_road_features.png', bbox_inches='tight')
plt.show()

---
## 📝 Step 8: Conclusion & Key Insights

In [ ]:
# Summary statistics to drive conclusion
peak_hr      = df['Hour'].value_counts().idxmax()
peak_month   = df['Month_Name'].value_counts().idxmax()
top_weather_cond = df['Weather_Condition'].value_counts().idxmax()
top_state    = df['State'].value_counts().idxmax()
top_city     = df['City'].value_counts().idxmax()
major_sev    = df['Severity'].value_counts().idxmax()
pct_sev2     = round(df['Severity'].value_counts(normalize=True)[2]*100, 1)

print('=' * 60)
print('  📋 KEY FINDINGS — US ACCIDENT DATA ANALYSIS')
print('=' * 60)
print(f'  🕐 Peak Accident Hour     : {peak_hr}:00 (rush hour)')
print(f'  📅 Highest Accident Month : {peak_month}')
print(f'  🌦️ Most Common Weather    : {top_weather_cond}')
print(f'  🏛️ Top State              : {top_state}')
print(f'  🏙️ Top City               : {top_city}')
print(f'  🚦 Dominant Severity      : Level {major_sev} ({pct_sev2}% of accidents)')
print('=' * 60)

## ✅ Conclusion

This analysis of the **US Accidents dataset (March 2023)** — covering **50,000 accident records** — reveals the following key insights:

| # | Insight | Finding |
|---|---------|--------|
| 1 | **Peak Time** | Most accidents occur during morning (7–9 AM) and evening (4–6 PM) rush hours |
| 2 | **Seasonal Trend** | Winter months (Nov–Jan) tend to show higher accident frequencies |
| 3 | **Weather** | Clear / Fair weather has the most accidents due to high traffic volume; fog and rain increase severity |
| 4 | **Severity** | The majority (~70–75%) of accidents are **Severity Level 2** (moderate), with very few Level 4 (critical) incidents |
| 5 | **Top States** | States like **California, Florida, and Texas** consistently report the highest accident counts |
| 6 | **Geographic Clusters** | Dense clusters are visible along **I-95 (East Coast)**, **I-5 (West Coast)**, and urban metro areas |
| 7 | **Road Features** | **Traffic Signals** and **Crossings** are most associated with accident locations |

### 🔮 Recommendations
- Increase law enforcement visibility during **7–9 AM** and **4–6 PM** rush hours
- Deploy additional safety measures at **traffic signals and crossings**
- Improve road infrastructure in high-accident states: **CA, FL, TX**
- Issue weather advisories during **fog and rain** to reduce high-severity incidents

---
*Analysis performed by: Saundarya Shembarkar | DS Internship Task-05*